# N3 - Substation current vs distributed voltage

**Question.** Fault *type* is encoded in the current sequence components at the feeder head
(`features.currents()`), while fault *location* lives in the spatial pattern of voltage dips
across buses. So: how much of what we call "placement value" could be replaced by **one
current measurement at the substation**?

**Why it matters.** This is the honest test of whether the paper's thesis survives the
obvious objection. If a feeder-head current gets most of the accuracy, then multi-bus voltage
placement is a location/robustness argument, not a *classification* argument — and the
manuscript must say so. If it does not, the thesis is stronger and directly evidenced.

**Configurations compared, at equal sensor count and equal noise:**

| config | what is measured |
|---|---|
| `V(k)` | k voltage snapshots (DMP-2 placement) — our method |
| `I_head` | substation current only, no voltage beyond the substation |
| `V(sub) + I_head` | substation voltage and current |
| `V(k) + I_head` | the k DMP buses plus the head current |
| `I_head` + `V(k-1)` | head current plus k-1 voltages (same total count) |

In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = []
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

## 1. Current-only and current-plus-voltage features

`features.currents()` gives sorted per-phase `|If|/|Ipre|`, `I0/I1`, `I2/I1` at the
feeder-head line. It applies the same measurement noise as the voltage path, so the
comparison is like-for-like.

In [ ]:
rows = []
feeders = K.feeder_list(SMOKE)

def cv(X, y, rf, seeds):
    return K.folds_acc(X, y, rf, seeds=seeds)

for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    head = F.head_line(ds)
    print(f"IEEE-{nb}: head line index {head} = {ds.lines[head]} between {ds.line_buses[head]}")

    for k in K.k_list(nb, SMOKE):
        obs, _ = Q.dmp2(m2, wgt, k)
        for noise in ("1% TVE", "uPMU"):
            seeds = (0,) if SMOKE else (0, 1)
            Xv = K.features_of(ds, obs, 0, noise, "FPS")
            rv = cv(Xv, ds.y, ds.rf, seeds)

            # current only, at the feeder head
            ms, asg = K.NOISE[noise]
            Xi = F.currents(ds, np.random.RandomState(0), mag_sigma=ms, ang_sigma_deg=asg)
            ri = cv(Xi, ds.y, ds.rf, seeds)

            # substation voltage only
            sub = ds.bus_ids([ds.substation])
            Xs = K.features_of(ds, sub, 0, noise, "FPS")
            rs = cv(Xs, ds.y, ds.rf, seeds)

            # substation V + head I (concatenated; both measured at one device)
            Xsi = np.hstack([Xs, Xi])
            rsi = cv(Xsi, ds.y, ds.rf, seeds)

            # k DMP voltages + head current
            Xvi = np.hstack([Xv, Xi])
            rvi = cv(Xvi, ds.y, ds.rf, seeds)

            # equal-count: head current + (k-1) voltages
            obs_1 = obs[1:] if len(obs) > 1 else obs
            Xv1 = K.features_of(ds, obs_1, 0, noise, "FPS")
            Xv1i = np.hstack([Xv1, Xi])
            rv1i = cv(Xv1i, ds.y, ds.rf, seeds)

            for name, r in (("V(k) DMP [ours]", rv), ("I_head only", ri),
                            ("V(sub) only", rs), ("V(sub)+I_head", rsi),
                            ("V(k)+I_head", rvi), ("I_head+V(k-1)", rv1i)):
                rows.append(dict(feeder=nb, k=k, noise=noise, config=name,
                                 acc=r["acc"], acc_sd=r["acc_sd"], hif=r["hif"]))
                print(f"  IEEE-{nb} k={k} {noise:8s} {name:18s} "
                      f"acc {r['acc']*100:5.1f}  hif {r['hif']*100:5.1f}")

import pandas as pd
df = pd.DataFrame(rows)
print("\n=== how much of the voltage-set accuracy does one head current buy? ===")
for (feeder, noise), g in df.groupby(["feeder", "noise"]):
    v = g[g.config == "V(k) DMP [ours]"]["acc"].mean()
    i = g[g.config == "I_head only"]["acc"].mean()
    vi = g[g.config == "V(k)+I_head"]["acc"].mean()
    print(f"IEEE-{feeder} {noise:8s}: V(k) {v*100:5.1f}%   I_head {i*100:5.1f}%   "
          f"V+I {vi*100:5.1f}%   (I_head recovers {100*(i-0.33)/(v-0.33):5.1f}% of the margin)")

## 2. Where does the head current lose? Fault resistance and DER

A head current senses a *sum* over the feeder, so it should be comparatively immune to which
bus faults but blind to the location. The next cell checks the `R_f` dependence, which is
where the physics predicts the two channels differ.

In [ ]:
detail = []
for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    for k in K.k_list(nb, SMOKE):
        obs, _ = Q.dmp2(m2, wgt, k)
        for noise in ("1% TVE", "uPMU"):
            ms, asg = K.NOISE[noise]
            rng = np.random.RandomState(0)
            Xs = {"V(k)": K.features_of(ds, obs, 0, noise, "FPS"),
                  "I_head": F.currents(ds, rng, mag_sigma=ms, ang_sigma_deg=asg)}
            for lo, hi in [(0, .1), (.1, 1), (1, 5), (5, 20), (20, 100)]:
                m = (ds.rf >= lo) & (ds.rf < hi)
                if m.sum() < 60:
                    continue
                for name, X in Xs.items():
                    r = K.folds_acc(X[m], ds.y[m], seeds=(0,))
                    detail.append(dict(feeder=nb, k=k, noise=noise, config=name,
                                       rf_lo=lo, rf_hi=hi, n=int(m.sum()),
                                       acc=r["acc"]))
                    print(f"IEEE-{nb} k={k} {noise:8s} R_f[{lo:5.2f},{hi:5.2f}) "
                          f"{name:7s} acc {r['acc']*100:5.1f}%")

## 3. Save

In [ ]:
import pandas as pd
pd.DataFrame(rows).to_csv(os.path.join(WORK, "N3_configs.csv"), index=False)
pd.DataFrame(detail).to_csv(os.path.join(WORK, "N3_rf_detail.csv"), index=False)
K.save_unit("n3", {"feeder": "all", "part": "main"},
            dict(rows=rows, detail=detail, meta=K.meta()))
print("TOTAL", round((time.time()-t_start)/60, 1), "min")